In [1]:
!pip install faiss-cpu
!pip install -U langchain-community
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.0/444.0 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.74
    Uninstalling langchain-core-0.3.74:
      Successfully uninstalled langchain-core-0.3.74
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which i

## 1️⃣ Initial Setup and Environment

**Task**: Set up the required libraries and check GPU availability.

**Requirements**:
- Import all necessary libraries for RAG implementation
- Check CUDA availability for GPU acceleration
- Set up proper logging and warning suppression

In [2]:
import torch
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
    pipeline
)
from datasets import load_dataset
import pandas as pd
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema import Document
import numpy as np
from tqdm import tqdm
import gc
import warnings
import os
import logging
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# Suppress warnings and logs
warnings.filterwarnings('ignore')
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
logging.getLogger("transformers").setLevel(logging.ERROR)


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2️⃣ Dataset Loading and Exploration

**Task**: Load the RAG dataset and explore its structure.

**Requirements**:
- Load the neural-bridge/rag-dataset-12000 dataset
- Convert to pandas DataFrame for easier manipulation
- Analyze the dataset structure and create a large document for retrieval
- Display basic statistics about the dataset

In [5]:
print("Loading RAG Dataset 12000...")
dataset = load_dataset("neural-bridge/rag-dataset-12000", split="test")

df = pd.DataFrame(dataset)
print(f"Dataset loaded with {len(df)} examples")

all_contexts = df['context'].unique()
large_document = "\n\n".join(all_contexts)
print(f"Combined document created with {len(large_document):,} characters")

Loading RAG Dataset 12000...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-9df3a936e1f631(…):   0%|          | 0.00/23.1M [00:00<?, ?B/s]

data/test-00000-of-00001-af2a9f454ad1b8a(…):   0%|          | 0.00/5.79M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9600 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2400 [00:00<?, ? examples/s]

Dataset loaded with 2400 examples
Combined document created with 8,250,984 characters


## 3️⃣ Text Chunking and Vector Store Setup

**Task**: Implement document chunking and create a vector store for retrieval.

**Options**:
- **Option A**: Use baseline parameters (chunk_size=124, overlap=10)
- **Option B**: Experiment with your own chunking strategy

**Requirements**:
- Choose and implement a text chunking strategy
- Set up embedding model for vectorization
- Create FAISS vector store from document chunks

In [6]:
print("Setting up text chunking and embeddings...")
text_splitter_A = RecursiveCharacterTextSplitter(
    chunk_size=124,
    chunk_overlap=10,
    length_function=len
)
chunks_A = text_splitter_A.split_text(large_document)
documents_A = [Document(page_content=chunk) for chunk in chunks_A]
embedding_model_A = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore_A = FAISS.from_documents(documents_A, embedding_model_A)

print("Vector store created successfully!")
print(f"Number of chunks: {len(documents_A)}")


Setting up text chunking and embeddings...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store created successfully!
Number of chunks: 87283


In [7]:
print("Setting up text chunking and embeddings")

# Custom RecursiveCharacterTextSplitter
text_splitter_B = RecursiveCharacterTextSplitter(
    chunk_size=256,
    chunk_overlap=50,
    length_function=len
)
chunks_B = text_splitter_B.split_text(large_document)
documents_B = [Document(page_content=chunk) for chunk in chunks_B]
embedding_model_B = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vectorstore_B = FAISS.from_documents(documents_B, embedding_model_B)

print("Vector store created successfully!")
print(f"Number of chunks: {len(documents_B)}")


Setting up text chunking and embeddings


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store created successfully!
Number of chunks: 46918


## 4️⃣ Language Model Selection and Setup

**Task**: Choose and set up a small language model for answer generation.

**Options**:
- **Option A**: Use baseline model (google/flan-t5-base)
- **Option B**: Choose your own model (<300M parameters)

**Baseline Configuration**:
- Model: google/flan-t5-base (~250M parameters)
- Temperature: 0.3
- Max length: 100 tokens

In [9]:
print("Loading small LLM for RAG system...")
model_name_A = "google/flan-t5-base"
tokenizer_A = AutoTokenizer.from_pretrained(model_name_A)
model_A = AutoModelForSeq2SeqLM.from_pretrained(
    model_name_A,
    torch_dtype=torch.float16,
    device_map="auto"
)
pipeline_A = pipeline(
    "text2text-generation",
    model=model_A,
    tokenizer=tokenizer_A,
    max_length=100,
    temperature=0.3,
    # device=0 if torch.cuda.is_available() else -1 # Removed the device argument
)
print("Small LLM pipeline created!")

Loading small LLM for RAG system...
Small LLM pipeline created!


## 5️⃣ Evaluation Framework Setup

**Task**: Set up the evaluation framework using cosine similarity-based assessment.

**Evaluation Method Explanation**:
Our evaluation uses cosine similarity between generated answers and ground truth with a 0.5 threshold. This approach:
- ✅ Captures semantic similarity between answers
- ✅ Works at scale without manual evaluation
- ❌ May miss exact factual correctness
- ❌ Threshold (0.5) is somewhat arbitrary
- ❌ Can give false positives for semantically similar but wrong answers

**Requirements**:
- Load sentence transformer for answer comparison
- Implement similarity-based evaluation function

In [10]:
print("Loading sentence transformer model for evaluation...")
eval_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("Model loaded!")
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
def evaluate_answer_correctness(generated_answer, ground_truth, threshold=0.5):
    """
    Evaluate answer correctness using cosine similarity.

    Limitations:
    - May not capture exact factual accuracy
    - Threshold is somewhat arbitrary
    - Short answers may have inflated scores
    - Paraphrases might score lower than expected
    """
    gen_clean = generated_answer.strip().lower()
    truth_clean = ground_truth.strip().lower()

    # Encode
    embeddings = eval_model.encode([gen_clean, truth_clean])

    # Calculate cosine similarity
    similarity_score = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    is_correct = similarity_score >= threshold
    return is_correct, similarity_score

Loading sentence transformer model for evaluation...
Model loaded!


In [15]:
#example
is_correct, score = evaluate_answer_correctness("ruam love RAG", "rag love ruam")
print(f"Correct: {is_correct}, Similarity: {score:.4f}")

Correct: True, Similarity: 0.9841


## 6️⃣ RAG Implementation

**Task**: Implement your RAG question-answering system.

**Baseline Approach** (if replicating):
- Retrieve top-2 most similar chunks
- Concatenate chunks as context
- Use prompt: "Answer the question based on the context. Context: {context} Question: {question}"
- Generate with temperature=0.3, max_length=50

**Your Approach** (if improving):
- Experiment with different retrieval strategies (k value, similarity thresholds)
- Try different prompt engineering approaches
- Consider context processing improvements
- Optimize generation parameters

In [16]:
def rag_answer_question(question, vectorstore, k=2):
    retrieved_docs = vectorstore.similarity_search(question, k=k)
    context = " ".join([doc.page_content for doc in retrieved_docs])
    prompt = f"Answer the question based on the context.\nContext: {context}\nQuestion: {question}"
    response = pipeline_A(
        prompt,
        max_length=50,
        temperature=0.3,
        num_return_sequences=1
    )

    return response[0]['generated_text']

In [17]:
test_row = df.iloc[0]
test_question = test_row['question']
ground_truth = test_row['answer']

# Generate an answer using RAG
generated_answer = rag_answer_question(test_question, vectorstore_B)  # Using Option B vectorstore

# Print results
print(f"Test Question: {test_question}")
print(f"Generated Answer: {generated_answer}")
print(f"Ground Truth: {ground_truth}")


Test Question: Who is the music director of the Quebec Symphony Orchestra?
Generated Answer: Fabien Gabel
Ground Truth: The music director of the Quebec Symphony Orchestra is Fabien Gabel.


## 7️⃣ Comprehensive Evaluation

**Task**: Evaluate your RAG system on the complete dataset.

**Baseline Performance to Match/Beat**:
- **Total Questions**: 2,399
- **Correct Answers**: 720  
- **Overall Accuracy**: 30.00%
- **Average Similarity**: 0.380
  - NOTE: some of the questions and answers were null, so be careful!!!
  - NOTE: The baseline might be a bit different for you as an LLM is not deterministic
    
**Requirements**:
- Run evaluation on the full dataset
- Track both accuracy and average similarity
- Handle errors gracefully
- Display progress and final results

In [18]:
from tqdm import tqdm
import pandas as pd
import numpy as np

def run_evaluation(dataset_df, vectorstore, num_samples=None, similarity_threshold=0.5):
    if num_samples is None:
        sample_df = dataset_df.copy()
    else:
        num_samples = min(num_samples, len(dataset_df))
        sample_df = dataset_df.sample(n=num_samples, random_state=42).reset_index(drop=True)
    results = []
    correct_count = 0
    total_similarity = 0.0
    evaluated_count = 0
    print("Starting RAG evaluation...")
    print(f"Evaluating on {len(sample_df)} samples...")
    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
        try:
            question = row.get('question', None)
            ground_truth = row.get('answer', None)
            if question is None or pd.isna(question):
                continue
            if ground_truth is None or pd.isna(ground_truth):
                continue
            generated_answer = rag_answer_question(question, vectorstore, k=2)
            if generated_answer is None or (isinstance(generated_answer, str) and generated_answer.strip() == ""):
                continue
            is_correct, sim = evaluate_answer_correctness(
                generated_answer,
                ground_truth,
                threshold=similarity_threshold
            )
            evaluated_count += 1
            if is_correct:
                correct_count += 1
            total_similarity += float(sim)

            results.append({
                "question": question,
                "ground_truth": ground_truth,
                "generated_answer": generated_answer,
                "similarity": float(sim),
                "is_correct": bool(is_correct)
            })

        except Exception as e:
            results.append({
                "question": row.get('question', None),
                "ground_truth": row.get('answer', None),
                "generated_answer": None,
                "similarity": None,
                "is_correct": False,
                "error": str(e)
            })
            continue
    final_accuracy = (correct_count / evaluated_count) if evaluated_count > 0 else 0.0
    avg_similarity = (total_similarity / evaluated_count) if evaluated_count > 0 else 0.0

    return results, final_accuracy, avg_similarity
# - Retrieval: vectorstore_B (Option B chunking you selected earlier)
# - Generator: flan-t5-base via pipeline_A (Option A LM)
results, final_accuracy, avg_similarity = run_evaluation(
    dataset_df=df,
    vectorstore=vectorstore_B,
    num_samples=None,
    similarity_threshold=0.5
)

# Display final results
total_questions = sum(1 for r in results if r.get("similarity") is not None)
correct_answers = sum(1 for r in results if r.get("is_correct") is True)

print("\n Evaluation Summary ")
print(f"Total Questions: {total_questions}")
print(f"Correct Answers: {correct_answers}")
print(f"Overall Accuracy: {final_accuracy*100:.2f}%")
print(f"Average Similarity: {avg_similarity:.3f}")

Starting RAG evaluation...
Evaluating on 2400 samples...


100%|██████████| 2400/2400 [25:21<00:00,  1.58it/s]


 Evaluation Summary 
Total Questions: 2399
Correct Answers: 1070
Overall Accuracy: 44.60%
Average Similarity: 0.461


In [20]:
print(f"\nFINAL RESULTS")
print(f"{'='*50}")
print(f"Total Questions Evaluated: {total_questions}")
print(f"Correct Answers: {correct_answers}")
print(f"Overall Accuracy: {final_accuracy*100:.2f}%")
print(f"Average Similarity Score: {avg_similarity:.3f}")
print(f"{'='*50}")


FINAL RESULTS
Total Questions Evaluated: 2399
Correct Answers: 1070
Overall Accuracy: 44.60%
Average Similarity Score: 0.461


## 📏 Evaluation Criteria

Your homework will be evaluated based on:

1. **Implementation Quality (40%)**
  - Correct RAG system implementation
  - Proper use of embedding models and vector stores
  - Valid model selection within constraints (<300M params)
  - Code quality and documentation

2. **Performance & Analysis (35%)**
  - Achieving reasonable performance (aim for >30% accuracy)
  - Thoughtful approach to optimization (if attempted)
  - Understanding of system components and their trade-offs

3. **Code Quality & Documentation (25%)**
  - Clean, readable, and well-structured code
  - Proper error handling and edge case management
  - Clear variable naming and function documentation
  - Efficient implementation and resource management